In [3]:
import sys

print("Python:", sys.version)
print("Intérprete:", sys.executable)
print("Entorno de Jupyter funcionando correctamente.")

Python: 3.12.1 (tags/v3.12.1:2305ca5, Dec  7 2023, 22:03:25) [MSC v.1937 64 bit (AMD64)]
Intérprete: c:\Users\Luis\Documents\Henry\M2\ProyectoM2_LuisFernandoCornejoMartínez\env1\Scripts\python.exe
Entorno de Jupyter funcionando correctamente.


# FleetLogix — Avance 1: Data Generation
**Proyecto Integrador Módulo 2 · Data Science · Henry**  
**Estudiante:** Luis Fernando Cornejo Martínez

> Este notebook **no genera ni inserta registros**. Documenta la generación realizada con el script independiente y ejecuta verificaciones de solo lectura sobre PostgreSQL. Los resultados deben ejecutarse y revisarse en el entorno del estudiante.

## 1. Objetivo y contexto
FleetLogix opera una flota logística. El objetivo del avance es estudiar su modelo relacional, generar datos sintéticos coherentes y comprobar su calidad.

**Volúmenes esperados:** 200 vehículos, 400 conductores, 50 rutas, 100 000 viajes, 400 000 entregas y 5 000 mantenimientos: **505 650 registros**.

**Herramientas:** PostgreSQL, Python, Faker, pandas, numpy, psycopg2, Jupyter y DBeaver.

**Importante:** la generación se realizó previamente mediante un script Python separado. No volver a ejecutar `main()` ni los scripts de carga sobre las tablas ya pobladas.

## 2. Modelo relacional
| Tabla | Llave primaria | Relaciones |
|---|---|---|
| `vehicles` | `vehicle_id` | Referenciada por `trips` y `maintenance` |
| `drivers` | `driver_id` | Referenciada por `trips` |
| `routes` | `route_id` | Referenciada por `trips` |
| `trips` | `trip_id` | FK hacia `vehicles`, `drivers`, `routes`; referenciada por `deliveries` |
| `deliveries` | `delivery_id` | FK `trip_id` hacia `trips` |
| `maintenance` | `maintenance_id` | FK `vehicle_id` hacia `vehicles` |

**Restricciones destacadas:** placas, códigos de empleado, licencias, códigos de ruta y números de seguimiento únicos; campos obligatorios y referencias entre tablas. Algunas FK del DDL original admiten `NULL`, por lo que se validan también las reglas de negocio.

### Diagrama ER (referencia textual)
```text
vehicles ──< trips >── drivers
               │
routes ────────┤
               └──< deliveries
vehicles ─────────< maintenance
```
**Pendiente para entrega:** insertar captura del diagrama ER profesional generado en DBeaver, sin alterar las tablas base.

## 3. Metodología de generación
El script independiente `01_data_generation.py` (o su archivo de trabajo original) genera datos sintéticos usando Faker, Python y librerías auxiliares; inserta respetando dependencias: primero tablas maestras, después viajes, entregas y mantenimientos.

Describir aquí, con base en el código final entregado:
- Semilla aleatoria y reproducibilidad, si se configuraron.
- Lógica de fechas, pesos y capacidades de vehículos.
- Distribución de entregas por viaje.
- Mantenimiento aproximadamente cada 20 viajes.
- Inserción por lotes, confirmaciones (`commit`) y manejo de errores.
- Registro de ejecución y duración de cargas.

**No se reproduce la ejecución del generador en este notebook para proteger la base ya cargada.**

## 4. Conexión segura de solo lectura
Definir las variables de entorno `FLEETLOGIX_HOST`, `FLEETLOGIX_PORT`, `FLEETLOGIX_DB`, `FLEETLOGIX_USER` y `FLEETLOGIX_PASSWORD`. No escribir ni subir contraseñas al notebook.

El código configura la transacción como **READ ONLY**; no crea ni modifica tablas.

In [4]:
import os
import getpass

os.environ["FLEETLOGIX_PASSWORD"] = getpass.getpass(
    "Contraseña de PostgreSQL: "
)

print("Contraseña configurada correctamente.")

Contraseña configurada correctamente.


In [5]:
import os
import pandas as pd
import psycopg2
from IPython.display import display

config = {
    "host": os.getenv("FLEETLOGIX_HOST", "localhost"),
    "port": int(os.getenv("FLEETLOGIX_PORT", "5432")),
    "dbname": os.getenv("FLEETLOGIX_DB", "fleetlogix"),
    "user": os.getenv("FLEETLOGIX_USER", "postgres"),
    "password": os.getenv("FLEETLOGIX_PASSWORD"),
}
if not config["password"]:
    raise RuntimeError("Configura FLEETLOGIX_PASSWORD como variable de entorno antes de conectar.")

conn = psycopg2.connect(**config)
conn.set_session(readonly=True, autocommit=True)

def consultar(sql):
    return pd.read_sql_query(sql, conn)

print("Conexión de solo lectura establecida.")

Conexión de solo lectura establecida.


## 5. Conteo de registros por tabla

In [6]:
sql_conteos = """
SELECT 'vehicles' AS tabla, COUNT(*) AS registros FROM public.vehicles
UNION ALL SELECT 'drivers', COUNT(*) FROM public.drivers
UNION ALL SELECT 'routes', COUNT(*) FROM public.routes
UNION ALL SELECT 'trips', COUNT(*) FROM public.trips
UNION ALL SELECT 'deliveries', COUNT(*) FROM public.deliveries
UNION ALL SELECT 'maintenance', COUNT(*) FROM public.maintenance;
"""
conteos = consultar(sql_conteos)
esperados = {"vehicles":200, "drivers":400, "routes":50, "trips":100000,
             "deliveries":400000, "maintenance":5000}
conteos["esperados"] = conteos["tabla"].map(esperados)
conteos["cumple"] = conteos["registros"] == conteos["esperados"]
display(conteos)
print("Total:", int(conteos["registros"].sum()))
print("Conteos correctos:", bool(conteos["cumple"].all()))

C:\Users\Luis\AppData\Local\Temp\ipykernel_14972\1957252154.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn)


,tabla,registros,esperados,cumple
0,vehicles,200,200,True
1,drivers,400,400,True
2,routes,50,50,True
3,trips,100000,100000,True
4,deliveries,400000,400000,True
5,maintenance,5000,5000,True


Total: 505650
Conteos correctos: True


## 6. Integridad referencial y claves foráneas
Las claves foráneas deben apuntar a registros existentes; también verificamos que no existan referencias nulas en los campos que exige la lógica del negocio.

In [7]:
sql_integridad = """
SELECT 'trips: vehicle_id inválido/nulo' AS prueba, COUNT(*) AS incidencias
FROM trips t LEFT JOIN vehicles v ON v.vehicle_id=t.vehicle_id
WHERE v.vehicle_id IS NULL
UNION ALL
SELECT 'trips: driver_id inválido/nulo', COUNT(*)
FROM trips t LEFT JOIN drivers d ON d.driver_id=t.driver_id WHERE d.driver_id IS NULL
UNION ALL
SELECT 'trips: route_id inválido/nulo', COUNT(*)
FROM trips t LEFT JOIN routes r ON r.route_id=t.route_id WHERE r.route_id IS NULL
UNION ALL
SELECT 'deliveries: trip_id inválido/nulo', COUNT(*)
FROM deliveries d LEFT JOIN trips t ON t.trip_id=d.trip_id WHERE t.trip_id IS NULL
UNION ALL
SELECT 'maintenance: vehicle_id inválido/nulo', COUNT(*)
FROM maintenance m LEFT JOIN vehicles v ON v.vehicle_id=m.vehicle_id WHERE v.vehicle_id IS NULL;
"""
display(consultar(sql_integridad))

C:\Users\Luis\AppData\Local\Temp\ipykernel_14972\1957252154.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn)


,prueba,incidencias
0,maintenance: vehicle_id inválido/nulo,0
1,trips: vehicle_id inválido/nulo,0
2,trips: driver_id inválido/nulo,0
3,trips: route_id inválido/nulo,0
4,deliveries: trip_id inválido/nulo,0


## 7. Unicidad de identificadores y campos de negocio

In [8]:
sql_unicidad = """
SELECT 'vehicles.license_plate' AS campo, COUNT(*) - COUNT(DISTINCT license_plate) AS duplicados,
       COUNT(*) FILTER (WHERE license_plate IS NULL) AS nulos FROM vehicles
UNION ALL SELECT 'drivers.employee_code', COUNT(*) - COUNT(DISTINCT employee_code),
       COUNT(*) FILTER (WHERE employee_code IS NULL) FROM drivers
UNION ALL SELECT 'drivers.license_number', COUNT(*) - COUNT(DISTINCT license_number),
       COUNT(*) FILTER (WHERE license_number IS NULL) FROM drivers
UNION ALL SELECT 'routes.route_code', COUNT(*) - COUNT(DISTINCT route_code),
       COUNT(*) FILTER (WHERE route_code IS NULL) FROM routes
UNION ALL SELECT 'deliveries.tracking_number', COUNT(*) - COUNT(DISTINCT tracking_number),
       COUNT(*) FILTER (WHERE tracking_number IS NULL) FROM deliveries;
"""
display(consultar(sql_unicidad))

C:\Users\Luis\AppData\Local\Temp\ipykernel_14972\1957252154.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn)


,campo,duplicados,nulos
0,vehicles.license_plate,0,0
1,drivers.employee_code,0,0
2,drivers.license_number,0,0
3,routes.route_code,0,0
4,deliveries.tracking_number,0,0


## 8. Coherencia temporal
Se revisan viajes con llegada anterior o igual a salida; entregas anteriores a la salida del viaje, posteriores a la llegada y entregas anteriores a su fecha programada. **Las últimas dos categorías no implican automáticamente un error:** pueden representar retrasos reales o adelantos; se interpretan según las reglas de negocio.

En revisiones anteriores se detectaron entregas posteriores a la llegada del viaje; conservar el análisis y explicar su tratamiento, en vez de ocultarlo.

In [9]:
sql_fechas = """
SELECT 'viajes llegada <= salida' AS prueba, COUNT(*) AS incidencias
FROM trips WHERE arrival_datetime IS NOT NULL AND arrival_datetime <= departure_datetime
UNION ALL
SELECT 'entregas antes de salida', COUNT(*)
FROM deliveries d JOIN trips t ON d.trip_id=t.trip_id
WHERE d.delivered_datetime < t.departure_datetime
UNION ALL
SELECT 'entregas después de llegada', COUNT(*)
FROM deliveries d JOIN trips t ON d.trip_id=t.trip_id
WHERE t.arrival_datetime IS NOT NULL AND d.delivered_datetime > t.arrival_datetime
UNION ALL
SELECT 'entregas después de programada', COUNT(*)
FROM deliveries WHERE delivered_datetime > scheduled_datetime
UNION ALL
SELECT 'licencias vencidas hoy', COUNT(*)
FROM drivers WHERE license_expiry IS NOT NULL AND license_expiry < CURRENT_DATE;
"""
display(consultar(sql_fechas))

C:\Users\Luis\AppData\Local\Temp\ipykernel_14972\1957252154.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn)


,prueba,incidencias
0,licencias vencidas hoy,0
1,viajes llegada <= salida,0
2,entregas después de programada,216516
3,entregas antes de salida,0
4,entregas después de llegada,12231


## 9. Distribución de entregas por viaje
La especificación pide 400 000 entregas para 100 000 viajes (promedio 4), con una distribución coherente. La consulta permite detectar viajes con menos de 2 o más de 6 entregas.

In [10]:
sql_distribucion = """
WITH entregas_por_viaje AS (
  SELECT t.trip_id, COUNT(d.delivery_id) AS total_entregas
  FROM trips t LEFT JOIN deliveries d ON d.trip_id=t.trip_id
  GROUP BY t.trip_id
)
SELECT COUNT(*) AS total_viajes, MIN(total_entregas) AS minimo,
       MAX(total_entregas) AS maximo,
       ROUND(AVG(total_entregas), 2) AS promedio,
       COUNT(*) FILTER (WHERE total_entregas < 2 OR total_entregas > 6) AS fuera_de_rango
FROM entregas_por_viaje;
"""
display(consultar(sql_distribucion))

C:\Users\Luis\AppData\Local\Temp\ipykernel_14972\1957252154.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn)


,total_viajes,minimo,maximo,promedio,fuera_de_rango
0,100000,4,4,4.0,0


## 10. Validación de pesos y capacidades
Revisar si los pesos son positivos, si el peso total del viaje supera la capacidad del vehículo y si la suma de paquetes supera el peso declarado para el viaje. Las diferencias pequeñas pueden deberse a redondeos y deben documentarse.

In [11]:
sql_pesos = """
WITH pesos_entregas AS (
 SELECT trip_id, SUM(package_weight_kg) AS suma_paquetes
 FROM deliveries GROUP BY trip_id
)
SELECT COUNT(*) FILTER (WHERE t.total_weight_kg <= 0) AS viajes_peso_no_positivo,
       COUNT(*) FILTER (WHERE t.total_weight_kg > v.capacity_kg) AS viajes_sobre_capacidad,
       COUNT(*) FILTER (WHERE p.suma_paquetes > t.total_weight_kg + 0.05) AS paquetes_superan_peso_viaje
FROM trips t JOIN vehicles v ON t.vehicle_id=v.vehicle_id
LEFT JOIN pesos_entregas p ON p.trip_id=t.trip_id;
"""
display(consultar(sql_pesos))

C:\Users\Luis\AppData\Local\Temp\ipykernel_14972\1957252154.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn)


,viajes_peso_no_positivo,viajes_sobre_capacidad,paquetes_superan_peso_viaje
0,0,0,0


## 11. Distribuciones estadísticas y patrones de negocio
Inspeccionar los tipos de vehículo, estados de viajes y entregas y frecuencia de mantenimientos. Evaluar si las proporciones resultan plausibles para una operación logística.

In [12]:
for nombre, sql in {
    "Tipos de vehículo": "SELECT vehicle_type, COUNT(*) AS total FROM vehicles GROUP BY vehicle_type ORDER BY total DESC",
    "Estado de viajes": "SELECT status, COUNT(*) AS total FROM trips GROUP BY status ORDER BY total DESC",
    "Estado de entregas": "SELECT delivery_status, COUNT(*) AS total FROM deliveries GROUP BY delivery_status ORDER BY total DESC",
    "Mantenimientos por vehículo": "SELECT MIN(n) AS minimo, MAX(n) AS maximo, ROUND(AVG(n),2) AS promedio FROM (SELECT v.vehicle_id, COUNT(m.maintenance_id) AS n FROM vehicles v LEFT JOIN maintenance m ON m.vehicle_id=v.vehicle_id GROUP BY v.vehicle_id) x"
}.items():
    print("\n", nombre)
    display(consultar(sql))


 Tipos de vehículo


C:\Users\Luis\AppData\Local\Temp\ipykernel_14972\1957252154.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn)


,vehicle_type,total
0,Van,69
1,Camión Grande,60
2,Camión Mediano,51
3,Motocicleta,20



 Estado de viajes


C:\Users\Luis\AppData\Local\Temp\ipykernel_14972\1957252154.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn)


,status,total
0,completed,99928
1,in_progress,72


... (2 outputs left out)

,delivery_status,total
0,delivered,399712
1,pending,288



 Mantenimientos por vehículo


C:\Users\Luis\AppData\Local\Temp\ipykernel_14972\1957252154.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn)


,minimo,maximo,promedio
0,0,40,25.0


### Evidencia: Diagrama entidad-relación

El siguiente diagrama representa las relaciones entre las
tablas principales de FleetLogix.

![Diagrama entidad-relación de FleetLogix](../diagrams/diagrama_er.png)


## 12. Registro de ejecución y evidencias

Las siguientes evidencias corresponden a las verificaciones realizadas sobre la base de datos FleetLogix mediante PostgreSQL, Python y Jupyter Notebook.

| Evidencia | Resultado / referencia |
|---|---|
| Fecha y entorno de ejecución | 08/11/2026 — Python 3.12.1, PostgreSQL, VS Code y Jupyter |
| Captura del diagrama ER | Diagrama incorporado en el notebook, antes de la sección 12 |
| Conteo de seis tablas | Sección 5: 505,650 registros; todos los conteos correctos |
| Integridad referencial | Sección 6: 0 incidencias en las cinco relaciones evaluadas |
| Unicidad de identificadores | Sección 7: 0 duplicados y 0 valores nulos |
| Coherencia temporal | Sección 8: 12,231 entregas posteriores a la llegada y 216,516 posteriores a la fecha programada |
| Distribución de entregas | Sección 9: 100,000 viajes con 4 entregas cada uno |
| Pesos y capacidades | Sección 10: 0 incidencias en las tres validaciones |
| Distribuciones estadísticas | Sección 11: 5,000 mantenimientos, promedio de 25 por vehículo |
| Log del generador y duración | Los archivos `data_generation.log` y `generation_summary.json` corresponden a ejecuciones históricas del 18/07/2026. El log registra un intento fallido y el JSON contiene conteos anteriores. La carga actual fue verificada mediante consultas SQL, obteniendo 505,650 registros. |

**Observación sobre la trazabilidad de los datos:** Durante la revisión se identificó que los archivos históricos de generación no coinciden completamente con el estado actual de la base de datos. Por esta razón, los resultados documentados en el presente notebook se sustentan en consultas de verificación ejecutadas directamente sobre PostgreSQL. No se modificaron los registros existentes durante esta validación.

### Conclusión del Avance 1

Se verificó la generación de **505,650 registros sintéticos**, distribuidos en seis tablas relacionadas de la base de datos FleetLogix. Los conteos coinciden con los volúmenes establecidos para el proyecto.

Las pruebas de integridad referencial no detectaron relaciones inválidas. Asimismo, las verificaciones de unicidad confirmaron la ausencia de identificadores duplicados y valores nulos en los campos evaluados.

Respecto a la coherencia temporal, se identificaron **12,231 entregas posteriores a la llegada del viaje** y **216,516 entregas posteriores a su fecha programada**. Estas situaciones requieren interpretación según las reglas operativas del negocio y se documentan como observaciones de calidad de datos.

La distribución de entregas es uniforme, con cuatro entregas por viaje. Las validaciones de pesos y capacidades no detectaron incidencias.

Finalmente, los registros de mantenimiento presentan un promedio de 25 intervenciones por vehículo, aunque existen vehículos sin mantenimientos registrados.

En conclusión, la base de datos cumple las verificaciones estructurales y cuantitativas realizadas. Las excepciones temporales y las particularidades de distribución deberán considerarse en los análisis posteriores.


## 13. Mejoras propuestas (sin modificar la estructura base)
- Definir reglas explícitas para ventanas temporales de entregas y viajes.
- Documentar índices adecuados para búsquedas por viaje, conductor y fecha.
- Evaluar restricciones `NOT NULL` adicionales en claves foráneas que el negocio requiere.
- Crear pruebas automatizadas de calidad de datos para futuras cargas.
- Incorporar bitácora de cargas y política de recuperación.

Estas son **propuestas**, no cambios ejecutados en PostgreSQL.

In [13]:
# Cerrar la conexión al terminar el análisis.
conn.close()
print("Conexión cerrada.")

Conexión cerrada.
